### Canibalización entre ítems de una vitrina

Se estima el parámetro de canibalización $\beta$ con los logs de la política random del Open Bandit Dataset. La idea es ver si un ítem pierde clics cuando comparte la vitrina con ítems de su misma categoría. El modelo que planteamos es

$$P(\text{clic en } k) = \theta_k(a_k) - \beta \cdot c_k,$$

donde $c_k$ es cuántos de los otros dos ítems de la vitrina son de la misma categoría que el ítem en la posición $k$. 

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

OBD_PATH = Path("Contenido/Datasets/open_bandit_dataset/open_bandit_dataset")
CAMPAIGN = "men"

In [ ]:
UF = [f"user_feature_{i}" for i in range(4)]


def leer_log(politica, campana):
    log = pd.read_csv(OBD_PATH / politica / campana / f"{campana}.csv",
                      usecols=["timestamp", "item_id", "position", "click"] + UF,
                      dtype={c: "category" for c in UF})
    ts = log["timestamp"].str.replace(r"(\d{2}:\d{2}:\d{2})\+", r"\1.000000+", regex=True)
    log["timestamp"] = pd.to_datetime(ts, format="%Y-%m-%d %H:%M:%S.%f%z")
    return log.sort_values("timestamp", kind="stable").reset_index(drop=True)


log = leer_log("random", CAMPAIGN)
pd.DataFrame({"Impresiones": [len(log)], "Timestamps distintos": [log["timestamp"].nunique()],
              "Ítems": [log["item_id"].nunique()], "CTR": [log["click"].mean()]})

,Impresiones,Timestamps distintos,Ítems,CTR
0,452949,439004,34,0.005124


## Reconstrucción de vitrinas

El dataset tiene una fila por impresión y no trae un identificador de vitrina, así que para saber qué ítems se mostraron juntos se agrupan las tres filas de cada vitrina.

Las filas de una misma vitrina no tienen exactamente el mismo timestamp (casi todos son distintos), pero aparecen seguidas, con el mismo usuario y separadas por milisegundos. Entonces se parte una vitrina nueva cuando:
- pasan más de $\tau$ segundos desde la fila anterior,
- cambian las características del usuario, o
- se repite una posición.

In [ ]:
def reconstruir_vitrinas(log, tau):
    gap = log["timestamp"].diff().dt.total_seconds().fillna(np.inf).to_numpy()
    mismo_usuario = (log[UF] == log[UF].shift()).all(axis=1).to_numpy()
    posiciones = log["position"].to_numpy()

    vitrina = np.empty(len(log), dtype=int)
    actual, vistas = -1, set()
    for i in range(len(log)):
        if gap[i] > tau or not mismo_usuario[i] or posiciones[i] in vistas:
            actual, vistas = actual + 1, set()
        vistas.add(posiciones[i])
        vitrina[i] = actual
    return vitrina


filas = []
for tau in (0.05, 0.1, 0.5):
    tamano = np.bincount(reconstruir_vitrinas(log, tau))
    filas.append({"tau (s)": tau, "Vitrinas": len(tamano),
                  "Con 1 fila": (tamano == 1).sum(), "Con 2 filas": (tamano == 2).sum(),
                  "Con 3 filas": (tamano == 3).sum(),
                  "Impresiones en vitrinas completas": np.mean(np.repeat(tamano, tamano) == 3)})
pd.DataFrame(filas)

,tau (s),Vitrinas,Con 1 fila,Con 2 filas,Con 3 filas,Impresiones en vitrinas completas
0,0.05,154748,4212,2871,147665,0.978024
1,0.10,153398,2633,1979,148786,0.985449
2,0.50,151976,1024,931,150021,0.993628


Con $\tau = 0{,}1$ el 98,5 % de las impresiones queda en vitrinas completas, y el resultado cambia poco con 0,05 o 0,5. Por lo tanto, nos quedamos con $\tau = 0{,}1$ y descartamos las vitrinas incompletas.

In [4]:
log["vitrina"] = reconstruir_vitrinas(log, 0.1)
completas = log[log.groupby("vitrina")["position"].transform("size") == 3].sort_values(["vitrina", "position"])

# una fila por vitrina, con el ítem y el clic de las posiciones 1, 2 y 3
vitrinas = completas["item_id"].to_numpy().reshape(-1, 3)
clics = completas["click"].to_numpy().reshape(-1, 3)
pd.DataFrame(vitrinas[:5], columns=["Posición 1", "Posición 2", "Posición 3"])

,Posición 1,Posición 2,Posición 3
0,0,23,25
1,25,30,17
2,25,30,17
3,25,30,17
4,0,23,25


## Vecinos de la misma categoría

Usamos como categoría de cada ítem la columna item_feature_1 del archivo item_context.csv. Para cada posición contamos cuántos de los otros dos ítems de la vitrina son de su misma categoría.

In [ ]:
def contar_vecinos(cat):

    # paso a int antes de sumar 
    igual_12 = (cat[:, 0] == cat[:, 1]).astype(int)
    igual_13 = (cat[:, 0] == cat[:, 2]).astype(int)
    igual_23 = (cat[:, 1] == cat[:, 2]).astype(int)
    return np.column_stack([igual_12 + igual_13, igual_12 + igual_23, igual_13 + igual_23])


item_context = pd.read_csv(OBD_PATH / "random" / CAMPAIGN / "item_context.csv", index_col=0).sort_values("item_id")
categoria = item_context["item_feature_1"].astype("category").cat.codes.to_numpy()
n_items = len(categoria)

vecinos = contar_vecinos(categoria[vitrinas])

pd.DataFrame({"Vecinos de la misma categoría": vecinos.ravel(), "click": clics.ravel()}) \
  .groupby("Vecinos de la misma categoría")["click"].agg(Impresiones="size", CTR="mean")

,Impresiones,CTR
Vecinos de la misma categoría,,
0,329195,0.004970
1,109036,0.004760
2,8127,0.004184


## Estimación de $\beta$

Estimo una regresión lineal del clic sobre el número de vecinos de la misma categoría, con un efecto fijo por cada combinación ítem × posición (34 × 3 = 102 en *men*):

$$r_{ik} = \alpha_{a_{ik},k} - \beta \cdot c_{ik} + \varepsilon_{ik}.$$

Los efectos fijos capturan qué tan atractivo es cada ítem en cada posición, así que $\beta$ sale de comparar un mismo ítem, en una misma posición, con distintos vecinos. Como en *random* los vecinos no dependen del usuario, esa comparación es causal.

Para estimar con efectos fijos se le resto a cada variable su promedio dentro de su celda ítem × posición y después hago una regresión simple. 

In [7]:
def estimar_beta(vitrinas, clics, vecinos, grupos, n_items):
    item, pos = vitrinas.ravel(), np.tile(np.arange(3), len(vitrinas))
    r, c = clics.ravel().astype(float), vecinos.ravel().astype(float)

    # le resto a r y a c su promedio dentro de cada celda ítem x posición
    celda = item * 3 + pos
    n_celda = np.bincount(celda, minlength=n_items * 3)
    r_dm = r - (np.bincount(celda, r, n_items * 3) / n_celda)[celda]
    c_dm = c - (np.bincount(celda, c, n_items * 3) / n_celda)[celda]
    pendiente = (c_dm * r_dm).sum() / (c_dm ** 2).sum()

    # error estándar agrupado: sumo los aportes de cada grupo antes de elevar al cuadrado
    residuo = r_dm - pendiente * c_dm
    aporte_grupo = np.bincount(np.repeat(grupos, 3), c_dm * residuo)
    se = np.sqrt((aporte_grupo ** 2).sum()) / (c_dm ** 2).sum()
    return -pendiente, se   # beta es la caída, por eso cambio el signo


grupos = conjunto.astype("category").cat.codes.to_numpy()
beta, se = estimar_beta(vitrinas, clics, vecinos, grupos, n_items)
ctr = clics.mean()

pd.Series({
    "beta": beta,
    "Error estándar": se,
    "IC 95% inferior": beta - 1.96 * se,
    "IC 95% superior": beta + 1.96 * se,
    "z": beta / se,
    "beta / CTR": beta / ctr,
    "Efecto mínimo detectable (80% potencia)": 2.8 * se,
}).round(5)

beta                                       0.00039
Error estándar                             0.00031
IC 95% inferior                           -0.00022
IC 95% superior                            0.00099
z                                          1.24430
beta / CTR                                 0.07866
Efecto mínimo detectable (80% potencia)    0.00087
dtype: float64

Por último reviso que el resultado no dependa de la ventana $\tau$. Para esto, y para lo que viene después, junto los pasos anteriores en dos funciones: una que, dado un log y su item_context, reconstruye las vitrinas y cuenta los vecinos, y otra que además estima $\beta$.

In [8]:
def vitrinas_campana(log, item_context, tau=0.1):
    categoria = item_context["item_feature_1"].astype("category").cat.codes.to_numpy()
    v = log.assign(vitrina=reconstruir_vitrinas(log, tau))
    v = v[v.groupby("vitrina")["position"].transform("size") == 3].sort_values(["vitrina", "position"])
    vit = v["item_id"].to_numpy().reshape(-1, 3)
    cl = v["click"].to_numpy().reshape(-1, 3)
    vec = contar_vecinos(categoria[vit])
    grp = pd.Series([tuple(sorted(x)) for x in vit]).astype("category").cat.codes.to_numpy()
    return vit, cl, vec, grp, categoria


def beta_campana(log, item_context, tau=0.1):
    vit, cl, vec, grp, categoria = vitrinas_campana(log, item_context, tau)
    b, s = estimar_beta(vit, cl, vec, grp, len(categoria))
    return {"Vitrinas": len(vit), "CTR": cl.mean(), "beta": b, "Error estándar": s}


pd.DataFrame([{"tau (s)": tau, **beta_campana(log, item_context, tau)} for tau in (0.05, 0.1, 0.5)]).round(6)

,tau (s),Vitrinas,CTR,beta,Error estándar
0,0.05,147665,0.004892,0.000363,0.000309
1,0.10,148786,0.004904,0.000386,0.000310
2,0.50,150021,0.004895,0.000382,0.000309


## Comparación entre campañas

Repito la estimación en la campaña *women*, que tiene más ítems (46) y casi el doble de impresiones, con la misma regla de reconstrucción ($\tau = 0{,}1$). La categoría vuelve a ser item_feature_1, ahora del item_context.csv de *women*.

In [9]:
filas, logs = [], {}
for campana in ("men", "women"):
    log_c = leer_log("random", campana)
    ic_c = pd.read_csv(OBD_PATH / "random" / campana / "item_context.csv", index_col=0).sort_values("item_id")
    logs[campana] = (log_c, ic_c)   # los guardo para usarlos más abajo
    res = beta_campana(log_c, ic_c)
    filas.append({"Campaña": campana, "Ítems": len(ic_c), "Categorías": ic_c["item_feature_1"].nunique(),
                  "Impresiones": len(log_c), **res})

comparacion = pd.DataFrame(filas).set_index("Campaña")
comparacion["IC 95% inferior"] = comparacion["beta"] - 1.96 * comparacion["Error estándar"]
comparacion["IC 95% superior"] = comparacion["beta"] + 1.96 * comparacion["Error estándar"]
comparacion["beta / CTR"] = comparacion["beta"] / comparacion["CTR"]
comparacion["Efecto mínimo detectable"] = 2.8 * comparacion["Error estándar"]
comparacion.round(5)

,Ítems,Categorías,Impresiones,Vitrinas,CTR,beta,Error estándar,IC 95% inferior,IC 95% superior,beta / CTR,Efecto mínimo detectable
Campaña,,,,,,,,,,,
men,34,7,452949,148786,0.00490,0.00039,0.00031,-0.00022,0.00099,0.07866,0.00087
women,46,10,864585,284694,0.00461,0.00001,0.00024,-0.00046,0.00048,0.00209,0.00067


## Sesgo por canibalización al evaluar *bts* con logs de *random*

En el benchmark de Saito et al. se estima el CTR de la política *bts* usando los logs de *random* y se compara con el CTR observado de *bts*. El IPW por posición corrige qué ítem aparece en cada posición, pero en los logs de *random* cada ítem aparece con los vecinos que eligió *random*, no con los que elegiría *bts*. Si *bts* junta más (o menos) ítems de la misma categoría que *random*, el IPW por posición tiene un sesgo $\beta\Delta$, con

$$\Delta = \sum_k \sum_a \pi_{\text{bts}}(a \mid k)\Big(\mathbb{E}_{\text{bts}}[c_k \mid a_k = a] - \mathbb{E}_{\text{random}}[c_k \mid a_k = a]\Big).$$

Es decir, para cada ítem en cada posición comparo cuántos vecinos de su categoría tuvo en *bts* y en *random*, ponderando por qué tan seguido lo muestra *bts* ahí. Todo esto se puede sacar de los logs:
- $\beta$ es el que estimé con *random*, donde los vecinos se asignan al azar. En *bts* no se puede estimar de forma causal, porque *bts* elige qué ítems mostrar juntos según cuáles le han funcionado.
- $\pi_{\text{bts}}(a \mid k)$ y $\mathbb{E}_{\text{bts}}[c_k \mid a_k = a]$ salen de las vitrinas de *bts*, reconstruidas con la misma regla.
- $\mathbb{E}_{\text{random}}[c_k \mid a_k = a]$ sale de las vitrinas de *random*.

Como *bts* va aprendiendo durante la semana, $\pi_{\text{bts}}$ representa su comportamiento promedio en el periodo. El sesgo $\beta\Delta$ está en clics por vitrina; al dividirlo por 3 queda en CTR por impresión, que es la escala de obp.

In [ ]:
def vecinos_por_item(vitrinas, vecinos, n_items):
    veces = np.column_stack([np.bincount(vitrinas[:, k], minlength=n_items) for k in range(3)])
    suma = np.column_stack([np.bincount(vitrinas[:, k], vecinos[:, k], n_items) for k in range(3)])
    return veces, suma / np.maximum(veces, 1)


filas = []
for campana, (log_r, ic_c) in logs.items():
    log_b = leer_log("bts", campana)
    vit_r, _, vec_r, _, categoria = vitrinas_campana(log_r, ic_c)
    vit_b, cl_b, vec_b, _, _ = vitrinas_campana(log_b, ic_c)
    n_items = len(categoria)

    _, vecinos_random = vecinos_por_item(vit_r, vec_r, n_items)
    veces_bts, vecinos_bts = vecinos_por_item(vit_b, vec_b, n_items)
    pi_bts = veces_bts / veces_bts.sum(axis=0)   

    en_bts = (pi_bts * vecinos_bts).sum()          # vecinos parecidos por vitrina con bts
    supuesto = (pi_bts * vecinos_random).sum()     # lo que asume el IPW por posición al usar random
    delta = en_bts - supuesto

    b, s = comparacion.loc[campana, "beta"], comparacion.loc[campana, "Error estándar"]
    ctr_bts = log_b["click"].mean()
    sesgos = sorted([(b - 1.96 * s) * delta / 3, (b + 1.96 * s) * delta / 3])
    filas.append({
        "Campaña": campana,
        "Impresiones bts": len(log_b),
        "Impresiones en vitrinas completas": 3 * len(vit_b) / len(log_b),
        "Vecinos parecidos por vitrina (bts)": en_bts,
        "Vecinos parecidos por vitrina (random, reponderado)": supuesto,
        "Delta": delta,
        "beta": b,
        "CTR bts (on-policy)": ctr_bts,
        "Sesgo (CTR por impresión)": b * delta / 3,
        "IC 95% inferior": sesgos[0],
        "IC 95% superior": sesgos[1],
        "Sesgo / CTR bts": b * delta / 3 / ctr_bts,
        "IC 95% inferior / CTR bts": sesgos[0] / ctr_bts,
        "IC 95% superior / CTR bts": sesgos[1] / ctr_bts,
    })
    del log_b 

sesgo_bts = pd.DataFrame(filas).set_index("Campaña").T
sesgo_bts

Campaña,men,women
Impresiones bts,4.077727e+06,7.765497e+06
Impresiones en vitrinas completas,9.722850e-01,9.669631e-01
Vecinos parecidos por vitrina (bts),9.619869e-01,6.053936e-01
"Vecinos parecidos por vitrina (random, reponderado)",8.764843e-01,6.963094e-01
Delta,8.550260e-02,-9.091575e-02
beta,3.857734e-04,9.613836e-06
CTR bts (on-policy),6.742727e-03,6.379373e-03
Sesgo (CTR por impresión),1.099488e-05,-2.913497e-07
IC 95% inferior,-6.324094e-06,-1.456078e-05
IC 95% superior,2.831385e-05,1.397808e-05


**Declaración de uso de IA**

Utilicé IA para que se entienda de forma correcta lo que hace cada parte del código y la construcción teórica matemática de cada segmento (markdowns) y para tratar inicialmente los datos (recuperar la información relevante al momento de cargar los datos). Además, se utilizó como apoyo en el código, para arreglar errores y hacer ajustes.